In [ ]:
#| default_exp diff

# diff

> Cell-level git diffs for notebooks

Notebook JSON is nearly unreadable in `git diff`: one changed cell shows up as a wall of escaped source, and metadata noise swamps the signal. The functions here diff notebooks at the cell level instead. Cells are matched by their id, so pairing is exact with no alignment heuristics, and each changed cell is reported as a small unified diff of its source. `render_diff` turns that into terminal-friendly output, and `nbdev_diff_driver` wires it into `git diff` itself (installed by `nbdev-install-hooks`).

In [ ]:
#| export
import json
from fastcore.utils import *
from fastcore.meta import delegates
from fastcore.script import *
from subprocess import CalledProcessError
from typing import Annotated
from fastgit import Git
from fastcore.nbio import *
from fastcore.nbio import _directive
from nbdev.doclinks import nbglob
from nbdev.diff_core import *

In [ ]:
import tempfile, random, io
from contextlib import redirect_stdout
from fastcore.test import *

In [ ]:
random.seed(42)

Everything below works against a demo repo holding one committed notebook plus some uncommitted edits:


In [ ]:
tmp = tempfile.TemporaryDirectory(prefix='nbdiff_test_')
td = Path(tmp.name)
g = Git(td)
g.init(b='main')
g('config', 'user.email', 'nbdev@fast.ai')
g('config', 'user.name', 'nbdev')


Its one committed notebook has five one-line cells, the first tagged `#| export`:

In [ ]:
nb_path = td/'test.ipynb'
nb = new_nb(['#| export\nx=1', 'y=2', 'a=3', 'b=4', 'c=5'])
write_nb(nb, nb_path)
g.add('test.ipynb')
g.commit(m='initial notebook')

[main (root-commit) 091874c] initial notebook
 1 file changed, 58 insertions(+)
 create mode 100644 test.ipynb

The uncommitted edits change the first cell's source and append a cell carrying a stored output and a meta-form `eval: false` directive:

In [ ]:
nb.cells[0].source = '#| export\nx = 100'
out = dict(output_type='execute_result', data={'text/plain':'101'}, metadata={}, execution_count=1)
nb.cells.append(mk_cell('x+1', outputs=[out], metadata={'nbdev': {'eval': 'false'}}))
write_nb(nb, nb_path)

In [ ]:
#| export
def read_nb_from_git(
    g:Git, # The git object
    path, # The path to the notebook (absolute or relative to git root)
    ref=None # The git ref to read from (e.g. HEAD); None for working dir
)->AttrDict: # The notebook; empty if `path` doesn't exist at `ref`
    "Read notebook from git ref (e.g. HEAD) at path, or working dir if ref is None"
    path = Path(path)
    if path.is_absolute(): path = path.relative_to(g.top())
    if ref is None: return read_nb(g.top()/path)
    try: raw = g.show(f'{ref}:{path}', raise_exc=True)
    except CalledProcessError as e:
        if " in '" not in str(e.stderr): raise
        return new_nb([])
    return dict2nb(json.loads(raw))

In [ ]:
read_nb_from_git(g, 'test.ipynb', 'HEAD').cells

[{'cell_type': 'code',
  'execution_count': None,
  'id': '390c8c7d',
  'metadata': {},
  'outputs': [],
  'source': '#| export\nx=1',
  'idx_': 0,
  'lang_': 'python'},
 {'cell_type': 'code',
  'execution_count': None,
  'id': '7247342c',
  'metadata': {},
  'outputs': [],
  'source': 'y=2',
  'idx_': 1,
  'lang_': 'python'},
 {'cell_type': 'code',
  'execution_count': None,
  'id': 'd8100f2f',
  'metadata': {},
  'outputs': [],
  'source': 'a=3',
  'idx_': 2,
  'lang_': 'python'},
 {'cell_type': 'code',
  'execution_count': None,
  'id': '6f770d65',
  'metadata': {},
  'outputs': [],
  'source': 'b=4',
  'idx_': 3,
  'lang_': 'python'},
 {'cell_type': 'code',
  'execution_count': None,
  'id': 'd670e58e',
  'metadata': {},
  'outputs': [],
  'source': 'c=5',
  'idx_': 4,
  'lang_': 'python'}]

A path that doesn't exist at `ref` (such as a notebook not yet committed) comes back as an empty notebook, so a new file diffs as "all cells added". Other git failures, such as a bad ref, still raise:

In [ ]:
test_eq(read_nb_from_git(g, 'new.ipynb', 'HEAD').cells, [])
test_fail(lambda: read_nb_from_git(g, 'test.ipynb', 'nosuchref'))

In [ ]:
#| export
def _src(c):
    "Source of cell `c` without its leading directive lines"
    dirs,code = c._partition()
    return ''.join([l for l in dirs if not _directive(l, c.lang_)] + code)

def _srcdict(nb, f=noop):
    "Dict of cell id->`f(cell)`, with positional fallback ids for pre-4.5 notebooks"
    return {c.get('id', f'c{c.idx_}'): f(c) for c in nb.cells}

def _nb_srcdict(g:Git, nb_path, ref=None, f=noop): return _srcdict(read_nb_from_git(g, nb_path, ref), f)

Cells are keyed by id. Notebooks saved before nbformat 4.5 have no cell ids, so their cells get positional `c<n>` keys instead, which still pair correctly as long as no cells were reordered:

In [ ]:
nb0 = new_nb(['a', 'b'])
for c in nb0.cells: c.pop('id')
test_eq(_srcdict(nb0, _src), {'c0': 'a', 'c1': 'b'})

In [ ]:
#| export
def _dtag(c):
    "Cell `c`'s merged directives as a ` [k k=v]` header tag, in `dir_tag`'s form; `''` if none"
    tag = ' '.join(k if not v else f'{k}={v}' for k,v in c.directives.items())
    return f' [{tag}]' if tag else ''


In [ ]:
#| export
def nbs_pair(
    nb_path, # Path to the notebook
    ref_a='HEAD', # First git ref (None for working dir)
    ref_b=None, # Second git ref (None for working dir)
    f=noop  # Function to call on contents
): # Tuple of two notebooks
    "NBs at two refs; None means working dir. By default provides HEAD and working dir"
    nb_path = Path(nb_path).resolve()
    g = Git(nb_path.parent)
    return _nb_srcdict(g, nb_path, ref_a, f), _nb_srcdict(g, nb_path, ref_b, f)

By default `nbs_pair` compares HEAD with the working directory, returning each version as `{id: cell}`. `f` maps the cells, so passing `_src` shows just the edited version's sources, without their directive lines:

In [ ]:
a,b = nbs_pair(nb_path)
nbs_pair(nb_path, f=_src)[1]


{'390c8c7d': 'x = 100',
 '7247342c': 'y=2',
 'd8100f2f': 'a=3',
 '6f770d65': 'b=4',
 'd670e58e': 'c=5',
 '0351d8ae': 'x+1'}

In [ ]:
#| export
def _cell_changes(
    nb_path, # Path to the notebook
    fn, # function to call to get dict values
    ref_a='HEAD', # First git ref (None for working dir)
    ref_b=None, # Second git ref (None for working dir)
    adds=True, # Include cells in b but not in a
    changes=True, # Include cells with different content
    dels=False, # Include cells in a but not in b
    metadata=False, # Consider cell metadata when comparing
    outputs=False # Consider cell outputs when comparing
): # Dict of results
    "Apply fn(cell_id, old_content, new_content) to changed cells between two refs"
    def cell_content(c):
        res = c.get('source', '')
        if metadata: res += '\n# metadata: ' + json.dumps(c.get('metadata', {}), sort_keys=True)
        if outputs: res += '\n# outputs: ' + json.dumps(c.get('outputs', []), sort_keys=True)
        return res
    old,new = nbs_pair(nb_path, ref_a, ref_b, f=cell_content)
    res = {}
    if adds: res |= {cid: fn(cid, '', new[cid]) for cid in new if cid not in old}
    if changes: res |= {cid: fn(cid, old[cid], new[cid]) for cid in new if cid in old and new[cid] != old[cid]}
    if dels: res |= {cid: fn(cid, old[cid], '') for cid in old if cid not in new}
    return res

@delegates(_cell_changes)
def changed_cells(nb_path, **kwargs):
    "Return set of cell IDs for changed/added/deleted cells between two refs"
    def f(cid,o,n): return cid
    return set(_cell_changes(nb_path, f, **kwargs).keys())

Added and modified cells are reported by default. Deleted cells are opt-in via `dels`, since they have no content in the new notebook to look at, and `metadata` and `outputs` widen what counts as a change. The demo repo's two edits:

In [ ]:
changed_cells(td/'test.ipynb')

{'0351d8ae', '390c8c7d'}

Each kind can be switched off; `changes=False` leaves just the added cell:

In [ ]:
test_eq(changed_cells(td/'test.ipynb', changes=False), {nb.cells[-1].id})

The shared `source_diff` function returns a unified diff. Notebook cells are source fragments rather than whole files, so `final_newline=False` ignores their trailing newline:


In [ ]:
print(source_diff('x = 1\ny=2', 'x = 100\ny=2', final_newline=False))

--- 
+++ 
@@ -1,2 +1,2 @@
-x = 1
+x = 100
 y=2


In [ ]:
#| export
@delegates(_cell_changes)
def cell_diffs(nb_path, **kwargs):
    "{cell_id:diff} for changed/added/deleted cells between two refs"
    def f(cid,o,n): return source_diff(o,n,final_newline=False)
    return _cell_changes(nb_path, f, **kwargs)

`cell_diffs` pairs each changed id with its source diff, so the modified cell shows its old and new line, and the added cell only new lines:

In [ ]:
d = cell_diffs(td/'test.ipynb')
d

{'0351d8ae': '--- \n+++ \n@@ -0,0 +1 @@\n+x+1',
 '390c8c7d': '--- \n+++ \n@@ -1,2 +1,2 @@\n #| export\n-x=1\n+x = 100'}

## Rendering

`render_diff` combines notebook cell headers, context cells, and stored outputs with the shared source renderer. The word matching and truncation algorithm lives in the hand-written `nbdev.diff_core` module with pytest tests in `tests/test_diff.py`. The algorithm has no notebook dependency; the examples below exercise notebook-specific behavior.


In [ ]:
#| export
OUT_MAXLEN = 512 # Most characters shown of a cell's stored output


Similar deleted and added lines merge into one `!` line. Word similarity excludes whitespace and punctuation, but the displayed changes retain both. Dissimilar lines stay separate `-` and `+` lines.

Without color, deletions use `[-…-]` and additions use `{+…+}`. With color, deletions are red and struck out, and additions are green:


In [ ]:
PrettyString(render_source_diff('x=1', 'x = 100', final_newline=False))


@@ -1 +1 @@
!x{+ +}=[-1-]{+ 100+}

In [ ]:
#| export
def _diff_order(old, new):
    "All cell ids in notebook order: `new`'s order, with deleted ids after their predecessor in `old`"
    res,olds = list(new),list(old)
    for i,cid in enumerate(olds):
        if cid in new: continue
        pred = first(o for o in reversed(olds[:i]) if o in res)
        res.insert(res.index(pred)+1 if pred else 0, cid)
    return res

Deleted cells aren't in the new notebook, so to place them in the output each one goes right after its nearest surviving predecessor from the old notebook. A notebook that is entirely deleted or entirely new keeps its own order:


In [ ]:
test_eq(_diff_order(dict.fromkeys('abcd'), dict.fromkeys('ace')), list('abcde'))
test_eq(_diff_order(dict.fromkeys('ab'), {}), list('ab'))
test_eq(_diff_order({}, dict.fromkeys('xy')), list('xy'))

In [ ]:
#| export
def _out_lines(c):
    "Stored outputs of cell `c` rendered as text, capped at `OUT_MAXLEN` chars, each line prefixed `| `"
    r = truncate(render_text(c.get('outputs', [])), OUT_MAXLEN)
    return [f'| {l}' for l in r.splitlines()]

def render_diff(
    old, # `{id: cell}` for the old version of the notebook
    new, # `{id: cell}` for the new version
    maxlen:int=MAXLEN, # Truncate diff lines to this width (falsy: no limit)
    color:bool=False, # Add ANSI colors?
    context:int=1, # Unchanged cells to show either side of each change
    line_context:int=3, # Unchanged source lines to show either side of each change
    show_out:bool=True # Show the stored outputs of changed and added cells?
): # One section per shown cell, in notebook order
    "Render cell-level changes between two notebooks as truncated unified diffs, with context cells and outputs"
    order = _diff_order(old, new)
    def _key(cid, d): return (_src(d[cid]), d[cid].directives) if cid in d else None
    def _changed(cid): return _key(cid, old)!=_key(cid, new)
    show = {j for i,cid in enumerate(order) if _changed(cid) for j in range(i-context, i+context+1)}
    res,prev = [],-1
    for i,cid in enumerate(order):
        if i not in show: continue
        if 0<=prev<i-1: res.append(render_line(f'## … {i-prev-1} cells', maxlen, color))
        prev = i
        c = new.get(cid) or old[cid]
        if not _changed(cid):
            lines = [render_line(l, maxlen, color) for l in [f'## {cid}{_dtag(c)}:'] + [' '+l for l in _src(c).splitlines()]]
        else:
            o,n = (_src(old[cid]) if cid in old else ''), (_src(new[cid]) if cid in new else '')
            kind = 'modified' if cid in old and cid in new else 'added' if cid in new else 'deleted'
            lines = [render_line(f'## {kind} {cid}{_dtag(c)}:', maxlen, color)]
            source = render_source_diff(o, n, maxlen=maxlen, color=color, context=line_context, final_newline=False)
            if source: lines.append(source)
            if show_out and cid in new: lines += [render_line(l, maxlen, color) for l in _out_lines(new[cid])]
        res.append('\n'.join(lines))
    return '\n\n'.join(res)

Each changed cell has a header with its change kind, id, and merged nbdev directives. Moving a directive between source and metadata makes no difference. Changing a directive produces a header even when the source is unchanged. Directive lines are omitted from the source diff.

The edit to `x` becomes one `!` line. Unchanged neighbours provide context, and a marker counts the cells skipped between hunks. The added cell's stored output follows its diff with a `|` prefix. Stored output is capped at `OUT_MAXLEN` characters:

In [ ]:
ids = list(b)
r = render_diff(a, b)
assert f'## modified {ids[0]} [export]:\n@@ -1 +1 @@\n!x{{+ +}}=[-1-]{{+ 100+}}' in r
assert f'## added {ids[-1]} [eval=false]:' in r
test_eq(render_diff({'d': mk_cell('#| export\nz=1')}, {'d': mk_cell('z=1', metadata={'nbdev': {'export': 'true'}})}), '')
mixed = mk_cell('#| export\nz=1', metadata={'nbdev': {'eval': 'false'}})
test_eq(render_diff({'d': mk_cell('z=1')}, {'d': mixed}), '## modified d [export eval=false]:')
assert f'## {ids[1]}:\n y=2' in r and f'## {ids[4]}:' in r and ids[2] not in r
assert '## … 2 cells' in r and '| 101' in r
assert '\x1b[34m## … 2 cells\x1b[0m' in render_diff(a, b, color=True)
PrettyString(r)

## modified 390c8c7d [export]:
@@ -1 +1 @@
!x{+ +}=[-1-]{+ 100+}

## 7247342c:
 y=2

## … 2 cells

## d670e58e:
 c=5

## added 0351d8ae [eval=false]:
@@ -0,0 +1 @@
+x+1
| 101

Set `color=True` to use ANSI colors instead of word-diff markers. Deleted cells retain their change label. Set `context=0` and `show_out=False` to show only changed source:

In [ ]:
rc = render_diff(a, b, color=True)
assert '\x1b[31;9m1\x1b[0m' in rc and '\x1b[32m+x+1\x1b[0m' in rc
assert '## deleted' in render_diff({'gone': mk_cell('x=9')} | a, b)
r0 = render_diff(a, b, context=0, show_out=False)
assert ids[1] not in r0 and '| 101' not in r0
PrettyString(r0)

## modified 390c8c7d [export]:
@@ -1 +1 @@
!x{+ +}=[-1-]{+ 100+}

## … 4 cells

## added 0351d8ae [eval=false]:
@@ -0,0 +1 @@
+x+1

In [ ]:
#| export
def nb_diff(
    nb_path, # Path to the notebook
    ref_a='HEAD', # First git ref
    ref_b=None, # Second git ref; None for working dir
    maxlen:int=MAXLEN, # Truncate diff lines to this width (falsy: no limit)
    color:bool=False, # Add ANSI colors?
    context:int=1, # Unchanged cells to show either side of each change
    line_context:int=3, # Unchanged source lines to show either side of each change
    show_out:bool=True # Show the stored outputs of changed and added cells?
): # Rendered diff of changed cells
    "Rendered cell diff for `nb_path` between two refs"
    a,b = nbs_pair(nb_path, ref_a, ref_b)
    return render_diff(a, b, maxlen=maxlen, color=color, context=context, line_context=line_context, show_out=show_out)

`maxlen` limits the visible width, not word matching. A long word-diff line shows the first change and nearby context, with elision markers counting omitted text at either end. Later changes appear only if they fit in the same window. `maxlen=0` shows complete lines. `context` counts notebook cells; `line_context` counts unchanged source lines within each changed cell:


In [ ]:
nb.cells[1].source = 'y = 2  # ' + ', '.join(f'note {i}' for i in range(30))
write_nb(nb, nb_path)
long_diff = nb_diff(nb_path)
assert all(len(l)<=MAXLEN for l in long_diff.splitlines())
assert any(re.search(r'…\[\d+\]$', l) for l in long_diff.splitlines())
assert nb.cells[1].source in nb_diff(nb_path, maxlen=0)
PrettyString(long_diff)

## modified 390c8c7d [export]:
@@ -1 +1 @@
!x{+ +}=[-1-]{+ 100+}

## modified 7247342c:
@@ -1 +1 @@
-y=2
+y = 2  # note 0, note 1, note 2, note 3, note 4, note 5, note 6, note 7, note 8, note 9, note 10, note 11, note 12, note 13, note 14, note 15, note 16, note 17, note 18, note…[93]

## d8100f2f:
 a=3

## … 1 cells

## d670e58e:
 c=5

## added 0351d8ae [eval=false]:
@@ -0,0 +1 @@
+x+1
| 101

## Git integration

`nbdev_diff_driver` is a Git external diff driver for notebooks and plain text. Git calls it once per changed file with the repo path and the old/new file contents (`/dev/null` for additions or deletions). Notebook paths use cell-level rendering; other paths use plain source rendering. Renames and mode changes remain visible even when their content is unchanged. `nbdev-install-hooks` registers notebooks, while other file types can be selected in your global Git attributes file:

```gitattributes
*.ipynb diff=jupyternotebook
*.md    diff=nbdev-prose
*.py    diff=nbdev-source
```

Named Git drivers share the same implementation with different display limits. For example, in your Git configuration:

```gitconfig
[diff "jupyternotebook"]
    command = nbdev-diff-driver --maxlen 180
[diff "nbdev-prose"]
    command = nbdev-diff-driver --maxlen 960
[diff "nbdev-source"]
    command = nbdev-diff-driver --maxlen 240
```

These are reading-oriented diffs. `git diff --no-ext-diff` bypasses the driver when you need an applicable patch. `--line-context` controls source-line context. `--context` controls notebook-cell context. `--no-show-out` hides stored notebook outputs. For an unmerged path, Git passes only its repo path and the driver reports it as unmerged.


In [ ]:
#| export
def _file_cells(path):
    "`{id: cell}` from an ipynb file; `{}` when missing or empty (e.g. `/dev/null`)"
    p = Path(path)
    if not p.exists() or not p.stat().st_size: return {}
    return _srcdict(read_nb(p))

_pos = Annotated[str, {'opt':False, 'nargs':'?'}]  # optional positional CLI arg, as git passes them

@call_parse
def nbdev_diff_driver(
    path:str, # Repo path of the file being diffed
    old_file:_pos=None, # Pre-change file (git temp file, or /dev/null)
    old_hex:_pos=None, # Pre-change blob hash
    old_mode:_pos=None, # Pre-change file mode
    new_file:_pos=None, # Post-change file
    new_hex:_pos=None, # Post-change blob hash
    new_mode:_pos=None, # Post-change file mode
    rename_to:_pos=None, # New repo path, when git detected a rename
    similarity:_pos=None, # Similarity score, when git detected a rename
    maxlen:int=MAXLEN, # Truncate diff lines to this width (0 for no limit)
    context:int=1, # Unchanged notebook cells to show either side of each change
    line_context:int=3, # Unchanged source lines to show either side of each change
    show_out:bool=True # Show the stored outputs of changed and added cells?
):
    "Git external diff driver for notebooks and plain text files"
    if not new_file: return print(f'unmerged: {path}')
    color = sys.stdout.isatty() or os.environ.get('GIT_PAGER_IN_USE')=='true'
    if Path(path).suffix=='.ipynb':
        res = render_diff(_file_cells(old_file), _file_cells(new_file), maxlen=maxlen, color=color,
                          context=context, line_context=line_context, show_out=show_out)
    else:
        old,new = Path(old_file).read_bytes(),Path(new_file).read_bytes()
        res = 'Binary files differ' if old!=new else ''
        if b'\0' not in old and b'\0' not in new:
            try: res = render_source_diff(old.decode('utf-8'), new.decode('utf-8'), maxlen=maxlen, color=color, context=line_context)
            except UnicodeDecodeError: pass
    meta = []
    if rename_to: meta.append(f'rename to {rename_to}')
    if old_mode and new_mode and old_mode!=new_mode: meta.append(f'mode {old_mode} -> {new_mode}')
    if not res and not meta: return
    print(render_line(f'# {path}', 0, color))
    for line in meta: print(render_line('# '+line, 0, color))
    if res: print(res)
    print()

We can call the driver directly the same way git does, with the old version extracted to a file:

In [ ]:
old_p = td/'old.ipynb'
old_p.write_text(g.show('HEAD:test.ipynb'))
nbdev_diff_driver('test.ipynb', str(old_p), '', '', str(nb_path), '', '')

# test.ipynb
## modified 390c8c7d [export]:
@@ -1 +1 @@
!x{+ +}=[-1-]{+ 100+}

## modified 7247342c:
@@ -1 +1 @@
-y=2
+y = 2  # note 0, note 1, note 2, note 3, note 4, note 5, note 6, note 7, note 8, note 9, note 10, note 11, note 12, note 13, note 14, note 15, note 16, note 17, note 18, note…[93]

## d8100f2f:
 a=3

## … 1 cells

## d670e58e:
 c=5

## added 0351d8ae [eval=false]:
@@ -0,0 +1 @@
+x+1
| 101



When sources and directives are unchanged, the driver prints nothing, including the path header. Stored outputs and other metadata do not count as changes:

In [ ]:
s = io.StringIO()
with redirect_stdout(s): nbdev_diff_driver('test.ipynb', str(old_p), '', '', str(old_p), '', '')
test_eq(s.getvalue(), '')

In [ ]:
#| export
@call_parse
def nbdev_diff(
    path:str=None, # Notebook or directory (default: project's notebooks folder)
    ref_a:str='HEAD', # First git ref
    ref_b:str=None, # Second git ref (default: working directory)
    maxlen:int=MAXLEN, # Truncate diff lines to this width (0 for no limit)
    color:bool_arg=None, # Add ANSI colors? (default: only if stdout is a tty)
    context:int=1, # Unchanged cells to show either side of each change
    line_context:int=3, # Unchanged source lines to show either side of each change
    show_out:bool=True # Show the stored outputs of changed and added cells?
):
    "Cell-level diffs for changed notebooks between two git refs"
    if color is None: color = sys.stdout.isatty()
    for p in sorted(nbglob(path)):
        res = nb_diff(p, ref_a, ref_b, maxlen=maxlen, color=color, context=context, line_context=line_context, show_out=show_out)
        if res: print(render_line(f'# {p}', 0, color), res, sep='\n', end='\n\n')

`nbdev-diff` is the standalone version, like a cell-level `git diff` for notebooks: it shows changes between two refs (or a ref and the working directory) for one notebook, a directory, or the whole project. Run from the demo repo, it renders the same pending changes, headed by the notebook path:


In [ ]:
with working_directory(td): nbdev_diff('test.ipynb', color=False)

# test.ipynb
## modified 390c8c7d [export]:
@@ -1 +1 @@
!x{+ +}=[-1-]{+ 100+}

## modified 7247342c:
@@ -1 +1 @@
-y=2
+y = 2  # note 0, note 1, note 2, note 3, note 4, note 5, note 6, note 7, note 8, note 9, note 10, note 11, note 12, note 13, note 14, note 15, note 16, note 17, note 18, note…[93]

## d8100f2f:
 a=3

## … 1 cells

## d670e58e:
 c=5

## added 0351d8ae [eval=false]:
@@ -0,0 +1 @@
+x+1
| 101



In [ ]:
g.add('test.ipynb')
g.commit(m='update notebook')
assert not changed_cells(td/'test.ipynb')
assert not cell_diffs(td/'test.ipynb')

In [ ]:
tmp.cleanup()

## export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()